# CVZX Compiler: Quick Start

This notebook introduces the current CVZX Compiler research prototype. It demonstrates structured CV-ZX diagrams, NetworkX graph conversion, and the initial graph rewrite foundations.

> **Prototype status:** identity, fusion, and chain-reduction foundations are implemented. The complete optimization pipeline, finite-squeezing semantics, and target-specific lowering remain under development.

## Compiler architecture

```text
CV circuit
    │
    ▼
Structured CV-ZX diagram
    │
    ▼
NetworkX graph representation
    │
    ▼
Identity, chain reduction, fusion
    │
    ▼
Fourier, displacement, terminal and copy-rule passes
    │
    ▼
Optimized CV-ZX graph
    │
    ▼
Optimized structured CV-ZX diagram
    │
    ▼
Target-specific lowering → CV simulator or real CV quantum computer
```

## Imports

Install the repository from its root directory before running the notebook:

```bash
python -m pip install -e ".[dev]"
```

In [ ]:
from importlib import import_module

import matplotlib.pyplot as plt
import networkx as nx

from cvzx.base_gates import CompositionDiagram, Fourier, PSpider, QSpider, TensorDiagram, ZxPoly
from cvzx.nx_graph import to_diagram, to_graph

nx_rules = import_module("cvzx.nx_rewrite_rules")
print("CVZX imported successfully.")
print("Active rewrite module:", nx_rules.__name__)

## Build a structured diagram

This example contains sequential composition, parallel composition, two identity spiders, and two compatible Q-spiders. It is intentionally small enough to understand while still showing the hierarchical representation.

In [ ]:
from cvzx.visualize_base_gates import visualize

zero = ZxPoly({})
phase_a = ZxPoly({1: 0.25})
phase_b = ZxPoly({1: 0.75})

identity_q = QSpider(1, 1, zero)
q_a = QSpider(1, 1, phase_a)
q_b = QSpider(1, 1, phase_b)
identity_p = PSpider(1, 1, zero)

main_row = CompositionDiagram([identity_q, q_a, q_b, identity_p])
second_row = Fourier()
diagram = TensorDiagram([main_row, second_row])

fig = visualize(diagram, title="Sample diagram")
plt.show()

## Convert to NetworkX

The graph IR stores operation types, arities, phases, structural metadata, and port-aware edge connectivity.

In [ ]:
graph = to_graph(diagram)

print(f"Nodes: {graph.number_of_nodes()}")
print(f"Edges: {graph.number_of_edges()}")
print(f"DAG: {nx.is_directed_acyclic_graph(graph)}")

for node_id, attrs in graph.nodes(data=True):
    summary = {
        key: attrs.get(key)
        for key in ("type", "kind", "n_inputs", "n_outputs", "phase", "container_type")
        if key in attrs
    }
    print(node_id, summary)

print("Port-aware edges:")
for source, target, attrs in graph.edges(data=True):
    ports = {key: attrs.get(key) for key in ("source_port", "target_port") if key in attrs}
    print(source, "->", target, ports)

In [ ]:
positions = nx.spring_layout(graph, seed=7)
labels = {
    node: f"{node}\n{attrs.get('type', attrs.get('container_type', 'node'))}"
    for node, attrs in graph.nodes(data=True)
}

plt.figure(figsize=(10, 6))
nx.draw_networkx(
    graph,
    pos=positions,
    labels=labels,
    node_color="#d8ecf3",
    edge_color="#4b5563",
    node_size=1800,
    font_size=8,
    arrows=True,
)
plt.title("Port-aware NetworkX representation")
plt.axis("off")
plt.show()

## Apply available rewrite rules

The helper below looks for the active NetworkX rewrite functions. It reports unavailable functions explicitly, because some higher-level optimization passes are still being implemented.

In [ ]:
from cvzx.nx_rewrite_rules import ChainReductionRule, FusionRule, GateRegister, IdentityRule

optimized_graph = graph.copy()
reg = GateRegister()

rules = [
    ("Identity", IdentityRule()),
    ("Fusion", FusionRule()),
    ("Chain reduction", ChainReductionRule()),
]

for label, rule in rules:
    before = (
        optimized_graph.number_of_nodes(),
        optimized_graph.number_of_edges(),
    )
    reg.build_from_graph(optimized_graph)
    rule.apply_rule(optimized_graph, reg)

    after = (
        optimized_graph.number_of_nodes(),
        optimized_graph.number_of_edges(),
    )

    print(f"{label}: nodes/edges {before} → {after}")

## Reconstruct the structured diagram

The final step in this quick start is to return from the graph IR to a structured diagram suitable for inspection and visualization.

In [ ]:
optimized_diagram = to_diagram(optimized_graph)
fig = visualize(optimized_diagram, title="Sample diagram Optimized")
plt.show()

## Visualizing rewrite rules

The dedicated visualization tests are currently the canonical examples for rendering diagrams and before/after rewrite results. Run them directly from the repository root:

```bash
python tests/test_visualize_base_gates.py
python tests/test_visualize_gates.py
```

In the future, these visualization workflows will be moved to dedicated Jupyter notebooks.

## Next steps

The next development stages are Fourier and displacement normalization, terminal and measurement absorption, copy rules, finite-squeezing and covariance/noise tracking, weighted CSUM arithmetic, and target-specific lowering to a CV simulator or real CV quantum computer.

## Reference

This independent implementation builds on: Hironari Nagayoshi, Warit Asavanant, Ryuhoh Ide, Kosuke Fukui, Atsushi Sakaguchi, Jun-ichi Yoshikawa, Nicolas C. Menicucci, and Akira Furusawa, *ZX graphical calculus for continuous-variable quantum processes*, Physical Review Research 7, 033141 (2025). [arXiv:2405.07246](https://arxiv.org/abs/2405.07246)